In [0]:
import ast
import json
from pathlib import Path
from typing import Any, Dict, Optional

_NOTEBOOK_MARKER_PREFIXES = ("# Databricks notebook source", "# COMMAND ----------")

# AST node types that are safe to keep: definitions only, never executable
# statements that would reach out to a real cluster/volume/widget.
_DEFINITION_NODE_TYPES = (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)


def _read_ipynb_source(ipynb_path: str) -> str:
    """Concatenate the source of every code cell in a .ipynb file (plain JSON, no nbformat needed)."""
    with open(ipynb_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    chunks = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = cell.get("source", "")
        text = "".join(source) if isinstance(source, list) else source
        chunks.append(text)
    return "\n\n".join(chunks)


def _strip_databricks_markers(source: str) -> str:
    """Drop Databricks' own magic comment lines; harmless to keep, but noisy in tracebacks."""
    lines = [
        line
        for line in source.splitlines()
        if not line.strip().startswith(_NOTEBOOK_MARKER_PREFIXES)
    ]
    return "\n".join(lines)


def _definitions_only(source: str) -> ast.Module:
    """Return an ast.Module containing only top-level import/function/class definitions."""
    tree = ast.parse(source)
    tree.body = [node for node in tree.body if isinstance(node, _DEFINITION_NODE_TYPES)]
    ast.fix_missing_locations(tree)
    return tree


def load_notebook_definitions(ipynb_path: str, injected_globals: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    path = Path(ipynb_path)
    if not path.exists():
        raise FileNotFoundError(f"Notebook not found: {ipynb_path}")

    raw_source = _read_ipynb_source(str(path))
    clean_source = _strip_databricks_markers(raw_source)
    tree = _definitions_only(clean_source)

    namespace: Dict[str, Any] = {"__name__": f"_notebook_under_test_{path.stem}"}
    if injected_globals:
        namespace.update(injected_globals)

    code = compile(tree, filename=str(path), mode="exec")
    exec(code, namespace)
    return namespace